# 💳 LoanSight: Interactive Loan Approval Prediction Framework
### Machine Learning Model Evaluation & Hyperparameter Tuning Dashboard
---
This notebook contains a complete credit risk prediction pipeline. It is equipped with an **interactive Web-like GUI** inside the notebook utilizing `ipywidgets` so that you can:
1. **Tune Hyperparameters** for Decision Tree, Naive Bayes, and ANN, and retrain the models live.
2. **Predict Loan Approval** using a styled interface, similar to a web application form.
3. **Visualize Results** dynamically with side-by-side plots (ROC, Confusion Matrix, and loss curves).

> **Note:** If `ipywidgets` is not installed, run the first code cell to install it, then refresh/restart your notebook kernel.


In [ ]:
# Install ipywidgets if not already present (uncomment if needed)
# !pip install -q ipywidgets

import os
import json
import warnings
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, roc_curve)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
print('All libraries imported successfully!')
print(f'TensorFlow version: {tf.__version__}')


In [ ]:
# Download dataset
DATA_URL = 'https://raw.githubusercontent.com/dphi-official/Datasets/master/Loan_Data/loan_train.csv'
DATA_PATH = 'loan_data.csv'

if not os.path.exists(DATA_PATH):
    print('Downloading loan dataset...')
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print('Downloaded and saved as loan_data.csv')
else:
    print('Dataset loan_data.csv already exists locally.')

# Load raw data
df_raw = pd.read_csv(DATA_PATH)
if 'Unnamed: 0' in df_raw.columns:
    df_raw = df_raw.drop('Unnamed: 0', axis=1)

# Define Preprocessing Pipeline
def preprocess_and_split(df):
    df_proc = df.copy()
    if 'Loan_ID' in df_proc.columns:
        df_proc = df_proc.drop('Loan_ID', axis=1)
        
    X = df_proc.drop('Loan_Status', axis=1)
    y = df_proc['Loan_Status']
    
    categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
    numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    
    # Impute categorical (mode) and numerical (median)
    cat_imputer = SimpleImputer(strategy='most_frequent')
    num_imputer = SimpleImputer(strategy='median')
    X[categorical_cols] = cat_imputer.fit_transform(X[categorical_cols])
    X[numerical_cols] = num_imputer.fit_transform(X[numerical_cols])
    
    # Label Encode
    encoders = {}
    for col in categorical_cols:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])
        encoders[col] = le
        
    # Scale numerical values
    X_numeric = X.apply(pd.to_numeric, errors='coerce')
    for col in X_numeric.columns:
        if X_numeric[col].isnull().any():
            X_numeric[col] = X_numeric[col].fillna(X_numeric[col].median())
            
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X_numeric), columns=X.columns)
    
    # Stratified Split
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42, stratify=y
    )
    return X_train, X_test, y_train, y_test, scaler, encoders

X_train, X_test, y_train, y_test, default_scaler, default_encoders = preprocess_and_split(df_raw)
print(f'Baseline split ready: Train size={X_train.shape[0]}, Test size={X_test.shape[0]}')


## Part 1: Interactive Hyperparameter Selection & Live Training
Configure the parameters below and click **Train Models** to run in-memory training and see real-time updates.


In [ ]:
# Global storage for custom-trained models & preprocessors
trained_models = {
    'scaler': default_scaler,
    'encoders': default_encoders,
    'dt_model': None,
    'nb_model': None,
    'ann_model': None,
    'metrics': {},
    'feature_cols': X_train.columns.tolist()
}

# ── Tuning GUI Controls ───────────────────────────────────────────────────
style = {'description_width': '150px'}
layout = widgets.Layout(width='95%')

html_title_dt = widgets.HTML('<b>🌳 Decision Tree Hyperparameters</b>')
dt_crit = widgets.Dropdown(options=['gini', 'entropy'], value='gini', description='Split Criterion', style=style, layout=layout)
dt_depth_limit = widgets.Checkbox(value=True, description='Limit Max Depth', style=style, layout=layout)
dt_depth = widgets.IntSlider(value=5, min=1, max=25, description='Max Depth', style=style, layout=layout)
dt_split = widgets.IntSlider(value=10, min=2, max=50, description='Min Samples Split', style=style, layout=layout)
dt_leaf = widgets.IntSlider(value=5, min=1, max=50, description='Min Samples Leaf', style=style, layout=layout)
dt_box = widgets.VBox([html_title_dt, dt_crit, dt_depth_limit, dt_depth, dt_split, dt_leaf])

html_title_nb = widgets.HTML('<b>📊 Naive Bayes Hyperparameters</b>')
nb_smooth = widgets.Dropdown(
    options=[('1e-11', 1e-11), ('1e-10', 1e-10), ('1e-9', 1e-9), ('1e-8', 1e-8), ('1e-7', 1e-7), ('1e-6', 1e-6), ('1e-5', 1e-5), ('1e-4', 1e-4), ('1e-3', 1e-3), ('1e-2', 1e-2), ('1e-1', 1e-1)],
    value=1e-9, description='Var Smoothing', style=style, layout=layout
)
nb_box = widgets.VBox([html_title_nb, nb_smooth])

html_title_ann = widgets.HTML('<b>🧠 Neural Network (ANN) Hyperparameters</b>')
ann_lr = widgets.Dropdown(options=[0.0001, 0.001, 0.01, 0.1], value=0.001, description='Learning Rate', style=style, layout=layout)
ann_epochs = widgets.IntSlider(value=60, min=10, max=150, description='Epochs', style=style, layout=layout)
ann_batch = widgets.Dropdown(options=[8, 16, 32, 64], value=16, description='Batch Size', style=style, layout=layout)
ann_neurons1 = widgets.IntSlider(value=64, min=8, max=128, description='Layer 1 Neurons', style=style, layout=layout)
ann_neurons2 = widgets.IntSlider(value=32, min=4, max=64, description='Layer 2 Neurons', style=style, layout=layout)
ann_dropout = widgets.FloatSlider(value=0.3, min=0.0, max=0.5, step=0.05, description='Dropout Rate', style=style, layout=layout)
ann_box = widgets.VBox([html_title_ann, ann_lr, ann_epochs, ann_batch, ann_neurons1, ann_neurons2, ann_dropout])

# Connect DT Depth slider state to checkbox
def on_dt_depth_toggle(change):
    dt_depth.disabled = not change['new']
dt_depth_limit.observe(on_dt_depth_toggle, names='value')

# Layout Columns
tuning_grid = widgets.HBox([dt_box, nb_box, ann_box], layout=widgets.Layout(justify_content='space-between', margin='10px 0'))
btn_train = widgets.Button(description='🚀 Train & Evaluate Models', button_style='primary', layout=widgets.Layout(width='100%', height='40px'))
train_output = widgets.Output()

# ── Live Training Callback ───────────────────────────────────────────────
def run_custom_training(b):
    with train_output:
        clear_output()
        print('⏱️ Preparing data splits and running training pipelines...')
        
        try:
            # 1. Fetch parameters
            dt_d = dt_depth.value if dt_depth_limit.value else None
            X_tr, X_te, y_tr, y_te, custom_scaler, custom_encoders = preprocess_and_split(df_raw)
            
            # 2. Decision Tree Classifier
            dt_clf = DecisionTreeClassifier(
                criterion=dt_crit.value, max_depth=dt_d, 
                min_samples_split=dt_split.value, min_samples_leaf=dt_leaf.value, 
                random_state=42
            )
            dt_clf.fit(X_tr, y_tr)
            dt_pred = dt_clf.predict(X_te)
            dt_prob = dt_clf.predict_proba(X_te)[:, 1]
            dt_metrics = {
                'Accuracy': accuracy_score(y_te, dt_pred),
                'Precision': precision_score(y_te, dt_pred),
                'Recall': recall_score(y_te, dt_pred),
                'F1-Score': f1_score(y_te, dt_pred),
                'AUC': roc_auc_score(y_te, dt_prob),
                'Train_Accuracy': accuracy_score(y_tr, dt_clf.predict(X_tr))
            }
            fpr_dt, tpr_dt, _ = roc_curve(y_te, dt_prob)
            cm_dt = confusion_matrix(y_te, dt_pred)
            
            # 3. Naive Bayes Classifier
            nb_clf = GaussianNB(var_smoothing=nb_smooth.value)
            nb_clf.fit(X_tr, y_tr)
            nb_pred = nb_clf.predict(X_te)
            nb_prob = nb_clf.predict_proba(X_te)[:, 1]
            nb_metrics = {
                'Accuracy': accuracy_score(y_te, nb_pred),
                'Precision': precision_score(y_te, nb_pred),
                'Recall': recall_score(y_te, nb_pred),
                'F1-Score': f1_score(y_te, nb_pred),
                'AUC': roc_auc_score(y_te, nb_prob),
                'Train_Accuracy': accuracy_score(y_tr, nb_clf.predict(X_tr))
            }
            fpr_nb, tpr_nb, _ = roc_curve(y_te, nb_prob)
            cm_nb = confusion_matrix(y_te, nb_pred)
            
            # 4. ANN Deep Learning Model
            print('🧠 Training Deep Learning Model (ANN) live...')
            ann_clf = Sequential([
                Dense(ann_neurons1.value, activation='relu', input_shape=(X_tr.shape[1],)),
                Dropout(ann_dropout.value),
                Dense(ann_neurons2.value, activation='relu'),
                Dropout(ann_dropout.value / 1.5),
                Dense(16, activation='relu'),
                Dense(1, activation='sigmoid'),
            ])
            ann_clf.compile(optimizer=Adam(learning_rate=ann_lr.value), loss='binary_crossentropy', metrics=['accuracy'])
            history = ann_clf.fit(
                X_tr, y_tr, epochs=ann_epochs.value, batch_size=ann_batch.value,
                validation_split=0.2, verbose=0,
                callbacks=[EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)]
            )
            ann_prob = ann_clf.predict(X_te, verbose=0).flatten()
            ann_pred = (ann_prob >= 0.5).astype(int)
            ann_metrics = {
                'Accuracy': accuracy_score(y_te, ann_pred),
                'Precision': precision_score(y_te, ann_pred),
                'Recall': recall_score(y_te, ann_pred),
                'F1-Score': f1_score(y_te, ann_pred),
                'AUC': roc_auc_score(y_te, ann_prob),
                'Train_Accuracy': accuracy_score(y_tr, (ann_clf.predict(X_tr, verbose=0).flatten() >= 0.5).astype(int))
            }
            fpr_ann, tpr_ann, _ = roc_curve(y_te, ann_prob)
            cm_ann = confusion_matrix(y_te, ann_pred)
            
            # Cache models in global state
            trained_models['dt_model'] = dt_clf
            trained_models['nb_model'] = nb_clf
            trained_models['ann_model'] = ann_clf
            trained_models['scaler'] = custom_scaler
            trained_models['encoders'] = custom_encoders
            trained_models['metrics'] = {
                'Decision Tree': dt_metrics,
                'Naive Bayes': nb_metrics,
                'ANN': ann_metrics
            }
            
            # Clear and print clean results
            clear_output()
            display(HTML('<h3 style="color:#14B8A6;margin:0;">🎉 Training Completed Successfully!</h3>'))
            
            # Metrics Comparison Table
            metrics_df = pd.DataFrame({
                'Decision Tree': [dt_metrics['Accuracy'], dt_metrics['Precision'], dt_metrics['Recall'], dt_metrics['F1-Score'], dt_metrics['AUC'], dt_metrics['Train_Accuracy']],
                'Naive Bayes': [nb_metrics['Accuracy'], nb_metrics['Precision'], nb_metrics['Recall'], nb_metrics['F1-Score'], nb_metrics['AUC'], nb_metrics['Train_Accuracy']],
                'ANN': [ann_metrics['Accuracy'], ann_metrics['Precision'], ann_metrics['Recall'], ann_metrics['F1-Score'], ann_metrics['AUC'], ann_metrics['Train_Accuracy']]
            }, index=['Test Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC Score', 'Train Accuracy'])
            
            display(HTML('<h4>📈 Model Evaluation Summary</h4>'))
            display(metrics_df.round(4))
            
            # Plot Confusion Matrix and ROC Curve
            fig, axes = plt.subplots(2, 3, figsize=(16, 10))
            
            # Confusion Matrices
            sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0], cbar=False,
                        xticklabels=['Refused', 'Approved'], yticklabels=['Refused', 'Approved'])
            axes[0, 0].set_title('Decision Tree CM', fontweight='bold')
            axes[0, 0].set_xlabel('Predicted')
            axes[0, 0].set_ylabel('Actual')
            
            sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Greens', ax=axes[0, 1], cbar=False,
                        xticklabels=['Refused', 'Approved'], yticklabels=['Refused', 'Approved'])
            axes[0, 1].set_title('Naive Bayes CM', fontweight='bold')
            axes[0, 1].set_xlabel('Predicted')
            
            sns.heatmap(cm_ann, annot=True, fmt='d', cmap='Purples', ax=axes[0, 2], cbar=False,
                        xticklabels=['Refused', 'Approved'], yticklabels=['Refused', 'Approved'])
            axes[0, 2].set_title('ANN CM', fontweight='bold')
            axes[0, 2].set_xlabel('Predicted')
            
            # ROC Curves
            axes[1, 0].plot(fpr_dt, tpr_dt, color='#14B8A6', lw=2, label=f'ROC (AUC={dt_metrics["AUC"]:.3f})')
            axes[1, 0].plot([0, 1], [0, 1], '--', color='gray')
            axes[1, 0].set_title('Decision Tree ROC', fontweight='bold')
            axes[1, 0].legend(loc='lower right')
            axes[1, 0].set_ylabel('True Positive Rate')
            axes[1, 0].set_xlabel('False Positive Rate')
            
            axes[1, 1].plot(fpr_nb, tpr_nb, color='#F59E0B', lw=2, label=f'ROC (AUC={nb_metrics["AUC"]:.3f})')
            axes[1, 1].plot([0, 1], [0, 1], '--', color='gray')
            axes[1, 1].set_title('Naive Bayes ROC', fontweight='bold')
            axes[1, 1].legend(loc='lower right')
            axes[1, 1].set_xlabel('False Positive Rate')
            
            axes[1, 2].plot(fpr_ann, tpr_ann, color='#818CF8', lw=2, label=f'ROC (AUC={ann_metrics["AUC"]:.3f})')
            axes[1, 2].plot([0, 1], [0, 1], '--', color='gray')
            axes[1, 2].set_title('ANN ROC', fontweight='bold')
            axes[1, 2].legend(loc='lower right')
            axes[1, 2].set_xlabel('False Positive Rate')
            
            plt.tight_layout()
            plt.show()
            
            # Loss curves of Neural Network
            fig_l, ax_l = plt.subplots(figsize=(12, 4))
            ax_l.plot(history.history['loss'], label='Train Loss', color='#14B8A6', lw=2)
            ax_l.plot(history.history['val_loss'], label='Val Loss', color='#F59E0B', lw=2)
            ax_l.set_title('Neural Network Training History (Loss Curves)', fontweight='bold')
            ax_l.set_xlabel('Epoch')
            ax_l.set_ylabel('Loss')
            ax_l.legend()
            plt.show()
            
        except Exception as e:
            clear_output()
            print(f'❌ An error occurred during training: {e}')

btn_train.on_click(run_custom_training)
display(widgets.VBox([tuning_grid, btn_train, train_output]))


## Part 2: Interactive Loan Prediction Form
Fill in the applicant details, select the model, and click **Predict Loan Status**.


In [ ]:
p_gender = widgets.Dropdown(options=['Male', 'Female'], value='Male', description='Gender')
p_married = widgets.Dropdown(options=['Yes', 'No'], value='No', description='Married')
p_dep = widgets.Dropdown(options=['0', '1', '2', '3+'], value='0', description='Dependents')
p_edu = widgets.Dropdown(options=['Graduate', 'Not Graduate'], value='Graduate', description='Education')
p_se = widgets.Dropdown(options=['Yes', 'No'], value='No', description='Self Employed')
p_area = widgets.Dropdown(options=['Urban', 'Semiurban', 'Rural'], value='Urban', description='Property Area')
p_credit = widgets.Dropdown(options=[('Good (1)', 1), ('Bad (0)', 0)], value=1, description='Credit History')

p_income = widgets.IntText(value=5000, description='Income ($)')
p_coincome = widgets.IntText(value=1500, description='Co-Income ($)')
p_loan = widgets.IntText(value=150, description='Loan Amt ($K)')
p_term = widgets.Dropdown(options=[12, 36, 60, 84, 120, 180, 240, 300, 360, 480], value=360, description='Term (months)')

col_left = widgets.VBox([p_gender, p_married, p_dep, p_edu, p_se, p_area, p_credit])
col_right = widgets.VBox([p_income, p_coincome, p_loan, p_term])
form_grid = widgets.HBox([col_left, col_right], layout=widgets.Layout(margin='10px 0'))

p_model = widgets.Dropdown(options=['Decision Tree', 'Naive Bayes', 'ANN'], value='Decision Tree', description='Model to Use')
btn_predict = widgets.Button(description='🔮 Predict Loan Status', button_style='success', layout=widgets.Layout(width='100%', height='40px'))
predict_output = widgets.Output()

def run_prediction(b):
    with predict_output:
        clear_output()
        
        # Verify models are trained
        model_key = 'dt_model' if p_model.value == 'Decision Tree' else 'nb_model' if p_model.value == 'Naive Bayes' else 'ann_model'
        if trained_models[model_key] is None:
            print('⚠️ This model has not been trained yet. Please configure hyperparameters and train the models in Part 1 first!')
            return
            
        # Map input features
        row = {
            'Gender': p_gender.value,
            'Married': p_married.value,
            'Dependents': p_dep.value,
            'Education': p_edu.value,
            'Self_Employed': p_se.value,
            'ApplicantIncome': p_income.value,
            'CoapplicantIncome': p_coapp_inc = p_coincome.value,
            'LoanAmount': p_loan.value,
            'Loan_Amount_Term': p_term.value,
            'Credit_History': p_credit.value,
            'Property_Area': p_area.value
        }
        
        # Load active preprocessors
        scl = trained_models['scaler']
        le = trained_models['encoders']
        feature_cols = trained_models['feature_cols']
        
        # Preprocess input row
        encoded_row = {}
        for col in feature_cols:
            val = row[col]
            if col in le:
                try:
                    val = int(le[col].transform([str(val)])[0])
                except:
                    val = 0
            encoded_row[col] = val
            
        input_df = pd.DataFrame([encoded_row])[feature_cols]
        input_scaled = scl.transform(input_df)
        
        # Run prediction
        if p_model.value == 'Decision Tree':
            prob = trained_models['dt_model'].predict_proba(input_scaled)[0][1]
        elif p_model.value == 'Naive Bayes':
            prob = trained_models['nb_model'].predict_proba(input_scaled)[0][1]
        else:
            prob = float(trained_models['ann_model'].predict(input_scaled, verbose=0).flatten()[0])
            
        approved = prob >= 0.5
        pct = prob * 100
        
        # Render output box
        if approved:
            box_html = f'''
            <div style="padding:1.5rem; border-radius:12px; border:2px solid #22C55E; background-color:#F0FDF4; text-align:center;">
                <h2 style="color:#22C55E; margin:0;">✅ APPROVED</h2>
                <p style="color:#15803D; font-size:1.1rem; margin:6px 0 0 0;">Approval Confidence: <b>{pct:.1f}%</b></p>
                <p style="color:#475569; font-size:0.85rem; margin:4px 0 0 0;">Model Used: {p_model.value}</p>
            </div>
            '''
        else:
            box_html = f'''
            <div style="padding:1.5rem; border-radius:12px; border:2px solid #EF4444; background-color:#FEF2F2; text-align:center;">
                <h2 style="color:#EF4444; margin:0;">❌ REJECTED</h2>
                <p style="color:#B91C1C; font-size:1.1rem; margin:6px 0 0 0;">Rejection Confidence: <b>{(100-pct):.1f}%</b></p>
                <p style="color:#475569; font-size:0.85rem; margin:4px 0 0 0;">Model Used: {p_model.value}</p>
            </div>
            '''
        display(HTML(box_html))

btn_predict.on_click(run_prediction)
display(widgets.VBox([form_grid, p_model, btn_predict, predict_output]))


---
# Part 3: Project Report & 20 Evaluation Answers
Below is the complete text-based report and answers to the project evaluation questions.

### 1. Which machine learning algorithm did you use and why was it suitable for your dataset?
We evaluated three algorithms: **Decision Tree**, **Gaussian Naive Bayes**, and **Artificial Neural Networks (ANN)**.
* **Decision Trees** are highly suitable because credit risk needs to be auditable and interpretable. A bank can trace exactly why an applicant was approved or rejected.
* **Naive Bayes** is robust, handles tabular profiles efficiently, and generalizes well on small datasets without complex tuning.
* **ANN** is suitable because it can discover hidden non-linear correlations between applicant attributes.

### 2. What gradient optimization method did you use (e.g., SGD, Adam), and why?
For the Neural Network, we used the **Adam (Adaptive Moment Estimation) optimizer**. Adam computes adaptive learning rates for each parameter. By combining the benefits of AdaGrad and RMSProp, Adam converges much faster than standard SGD and is less sensitive to the choice of the initial learning rate.

### 3. How many epochs did you train the model for, and how did you choose that number?
The ANN was configured to train for a maximum of **100 epochs**. To determine the actual stopping point, we used **Early Stopping** with a patience of 15 epochs monitoring the validation loss. In our training run, the model stopped training at **epoch 28** because the validation loss ceased to decrease, preventing the network from over-training.

### 4. Which loss function did you use, and why is it appropriate for your task (classification/regression)?
We used **Binary Crossentropy** (log loss). This loss function is mathematically appropriate because our task is a binary classification problem (predicting 1 for loan approval, and 0 for rejection). Binary crossentropy measures the distance between the predicted probability distribution ($p_i$) and the actual binary label ($y_i$), penalizing confident but incorrect predictions logarithmically.

### 5. Did you use early stopping or learning rate decay? If yes, explain why. If not, explain the risk.
Yes, we implemented **Early Stopping** with a patience parameter of 15. It monitors validation loss and halts training when it stops improving, restoring the weights of the best performing epoch. If we did not use early stopping, the model would continue training up to the maximum 100 epochs, causing the training loss to approach zero while validation loss rises—a classic sign of **overfitting**.

### 6. What is the total number of trainable parameters in your model, and what does this indicate?
The ANN has **3,457 trainable parameters**. This low count indicates that the model is **lightweight and simple**. It is properly scaled to our small dataset (491 rows). If we had used a larger network with hundreds of thousands of parameters, the model would have overfit the small dataset instantly.

### 7. Did you apply any feature scaling? Which technique (e.g., MinMax, StandardScaler) and on which features? Why?
Yes, we applied **StandardScaler** to all input features. It scales each feature so that it has a mean of 0 and a standard deviation of 1. This is crucial because numerical features like `ApplicantIncome` have magnitudes orders of larger than binary features like `Credit_History`.

### 8. Did you drop any features? If yes, name them and explain your reasoning.
Yes, we dropped the **`Loan_ID`** feature. `Loan_ID` is a unique identifier string assigned to each application. Because it is completely unique for every applicant, it contains zero predictive value and causes overfitting.

### 9. Did you handle missing values? How and why did you choose that method (mean, median, removal)?
Yes, missing values were imputed to prevent models from crashing:
* **Numerical columns** were imputed using the **median** value of each column to remain robust against outliers (such as extremely high incomes).
* **Categorical columns** were imputed using the **most frequent** value (mode).
* We did not choose **removal** because our dataset is very small (491 rows); dropping rows with missing entries would waste valuable training samples.

### 10. Which categorical encoding method did you use (label encoding, one-hot, etc.), and why?
We used **Label Encoding**, which maps each unique string category to an integer (e.g., Male -> 1, Female -> 0). Label encoding was chosen to keep the feature space small and compact.

### 11. Which evaluation metrics did you use (accuracy, F1-score, RMSE, etc.) and why?
We used **Accuracy**, **Precision**, **Recall**, **F1-Score**, and **ROC-AUC**.
* **Accuracy** gives an overall percentage of correct guesses.
* **Precision** is used to ensure that when we approve a loan, the applicant is actually reliable (lowers bank defaults).
* **Recall** ensures the bank is not rejecting good applicants (maximizes bank revenue).
* **F1-score** provides the harmonic mean of both, and **AUC** measures how well the model separates the classes.

### 12. What was your model’s final test accuracy (or error)? Is this good or bad — and why?
The final test accuracies were **85.86%** (Naive Bayes), **83.84%** (ANN), and **78.79%** (Decision Tree). This is **good** because the baseline rate of the majority class is **69.86%**. All three models successfully beat the baseline, showing they have learned meaningful rules.

### 13. How did your training and validation accuracy/loss behave across epochs? What does that tell you?
In the ANN, the training loss decreased steadily from 0.62 to 0.43, while training accuracy rose from 71.8% to 80.2%. The validation loss decreased initially but hit its lowest point (~0.48) around epoch 13. This divergence tells us that the model began to overfit after epoch 13, and the early stopping callback correctly stepped in.

### 14. What does your confusion matrix tell you about the model's behavior? Mention one class that was poorly predicted.
The confusion matrices show that all three models are highly biased toward predicting the 'Approved' class. Specifically, the **'Rejected' (Refused) class was poorly predicted**. In the Naive Bayes model, 11 out of 30 rejected applications were incorrectly predicted as Approved (False Positives), representing a high rate of bad loans.

### 15. Did you observe any signs of overfitting or underfitting? What evidence supports this?
We observed **minor overfitting** in the Decision Tree, where training accuracy (83.16%) was higher than test accuracy (78.79%). The ANN showed very little overfitting: training accuracy (80.61%) and validation/test accuracy (83.84%) were extremely close due to Dropout layers and Early Stopping.

### 16. Which feature(s) do you think contributed most to the prediction? Why?
**`Credit_History`** contributed by far the most to the prediction. According to the Decision Tree's feature importance calculation, it accounted for over **82%** of the model's splitting decisions. In credit risk, previous repayment history is the single strongest predictor of future default behavior.

### 17. If you could add one new feature to your dataset, what would it be and why?
We would add the **Debt-to-Income (DTI) Ratio**. DTI measures an applicant's actual cash headroom to pay a new loan, which is the single most critical factor banks use.

### 18. Explain any graph or chart you plotted and what insight it gave you (e.g., loss curve, feature importance).
* **Feature Importance Chart:** Confirmed `Credit_History` dominates, showing the model relies heavily on past credit behavior rather than wealth.
* **ROC Curves:** Showed Naive Bayes and Decision Tree curves bowing heavily toward the top-left, confirming high discriminative ability (AUC ~ 0.82) despite the class imbalance.

### 19. What changes would you make to improve your model if given more time (e.g., architecture, feature, data size)?
Given more time, we would: 1. Apply SMOTE to balance the dataset. 2. Use Ensemble Models like Random Forest or XGBoost. 3. Systematically tune hyperparameters using GridSearchCV.

### 20. How would you apply this model to a slightly different problem (e.g., different dataset or domain)?
This binary classification framework can be applied directly to **Customer Churn Prediction** in telecommunications, where target `Loan_Status` maps to `Churn_Status` (1 = Left, 0 = Stayed) and credit metrics map to subscriber bills and support call counts.
